### 데이터 분석 컨설팅 20240703
* 해운대구청 기획예산과
* 모래축제 기간 전(2주전)과 기간 내 유동인구와 매출현황 비교
* 비교기간: 5. 10.-13.(축제 2주전), 5. 24.-27.(축제)
* 축제구간: (1) 해운대광장(구남로), (2) 해운대시장, (3) 해리단길, (4) 해운대해수욕장
* 성별, 연령별, 타지역, 외국인 분석

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import PublicDataReader as pdr

study_area = gpd.read_file("project/26350_20240703.gpkg", layer = "study_area", driver="GPKG")
area_grid = gpd.read_file("project/26350_20240703.gpkg", layer = "area_grid", driver="GPKG")
area_grid['x'] = area_grid.centroid.x
area_grid['y'] = area_grid.centroid.y


/opt/conda/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [3]:
hcode = pdr.code_hdong()
hcode = hcode.loc[(hcode['시군구명']!='') & (hcode['읍면동명']!='') & (hcode['말소일자']=='')]
hcode = hcode[['시도코드','시도명','시군구코드','시군구명']]
hcode.columns = ['scode', 'sname', 'gcode', 'gname']

출처: 행정기관(행정동) 및 관할구역(법정동) 변경내역(2024.8.1. 시행)


#### 성연령별 생활인구 집계

In [15]:

df = pd.read_csv("D:/10_분석업무/구군청/26350_해운대구/2024년/20240704/생활인구/skt_d_service_sex_age_p_202405.csv", sep="|", 
                 parse_dates=['std_ymd'], dtype={'hcode': object, 'id': object}, encoding="utf-8", low_memory=False)
df = df.loc[(df['std_ymd'].dt.day >= 10) & (df['std_ymd'].dt.day <= 13) | (df['std_ymd'].dt.day >= 24) & (df['std_ymd'].dt.day <= 27) & (df['hcode'].str.startswith('26350'))]
df = pd.merge(df, area_grid, "inner", "id")

df.loc[df['std_ymd'].astype(str).str.contains('2024-05-10|2024-05-11|2024-05-12|2024-05-13'), 'peroid'] = "축제2주전"
df.loc[df['std_ymd'].astype(str).str.contains('2024-05-24|2024-05-25|2024-05-26|2024-05-27'), 'peroid'] = "축제기간"

df['h_m_sum'] = df.loc[:, 'h_m_0009':'h_m_7000'].sum(axis=1)
df['h_w_sum'] = df.loc[:, 'h_w_0009':'h_w_7000'].sum(axis=1)
df['w_m_sum'] = df.loc[:, 'w_m_0009':'w_m_7000'].sum(axis=1)
df['w_w_sum'] = df.loc[:, 'w_w_0009':'w_w_7000'].sum(axis=1)
df['v_m_sum'] = df.loc[:, 'v_m_0009':'v_m_7000'].sum(axis=1)
df['v_w_sum'] = df.loc[:, 'v_w_0009':'v_w_7000'].sum(axis=1)
df['m_sum'] = df['h_m_sum'] + df['w_m_sum'] + df['v_m_sum']
df['w_sum'] = df['h_w_sum'] + df['w_w_sum'] + df['v_w_sum']

df['10less'] = df.loc[:, ['h_m_0009','h_w_0009','w_m_0009','w_w_0009','v_m_0009','v_w_0009']].sum(axis=1)
df['10ages'] = df.loc[:, ['h_m_1014','h_m_1519','h_w_1014','h_w_1519','w_m_1014','w_m_1519','w_w_1014','w_w_1519','v_m_1014','v_m_1519','v_w_1014','v_w_1519']].sum(axis=1)
df['20ages'] = df.loc[:, ['h_m_2024','h_m_2529','h_w_2024','h_w_2529','w_m_2024','w_m_2529','w_w_2024','w_w_2529','v_m_2024','v_m_2529','v_w_2024','v_w_2529']].sum(axis=1)
df['30ages'] = df.loc[:, ['h_m_3034','h_m_3539','h_w_3034','h_w_3539','w_m_3034','w_m_3539','w_w_3034','w_w_3539','v_m_3034','v_m_3539','v_w_3034','v_w_3539']].sum(axis=1)
df['40ages'] = df.loc[:, ['h_m_4044','h_m_4549','h_w_4044','h_w_4549','w_m_4044','w_m_4549','w_w_4044','w_w_4549','v_m_4044','v_m_4549','v_w_4044','v_w_4549']].sum(axis=1)
df['50ages'] = df.loc[:, ['h_m_5054','h_m_5559','h_w_5054','h_w_5559','w_m_5054','w_m_5559','w_w_5054','w_w_5559','v_m_5054','v_m_5559','v_w_5054','v_w_5559']].sum(axis=1)
df['60ages'] = df.loc[:, ['h_m_6064','h_m_6569','h_w_6064','h_w_6569','w_m_6064','w_m_6569','w_w_6064','w_w_6569','v_m_6064','v_m_6569','v_w_6064','v_w_6569']].sum(axis=1)
df['70more'] = df.loc[:, ['h_m_7000','h_w_7000','w_m_7000','w_w_7000','v_m_7000','v_w_7000']].sum(axis=1)

gender = np.round(df.groupby(['peroid','id','areanm'])[['m_sum', 'w_sum']].sum().reset_index().sort_values(by=['peroid','areanm']))
gender = pd.melt(gender, id_vars=['peroid', 'id', 'areanm'], value_vars=['m_sum', 'w_sum'], var_name='class', value_name='pop')
gender.loc[gender['class'].str.startswith('m'), 'gender'] = '남성'
gender.loc[gender['class'].str.startswith('w'), 'gender'] = '여성'
gender = gender[['peroid', 'id', 'areanm', 'gender', 'pop']]
print("===== 성별 생활인구 밀도 시각화용 데이터 =====")
gender = pd.merge(gender, area_grid, "inner", "id")
gender = pd.DataFrame(gender.drop(columns=['areanm_y','geometry']))
gender.columns = ['peroid','id','areanm','gender','pop','x','y']
geom = gpd.points_from_xy(gender.x, gender.y, crs="EPSG:5179")
gender = gpd.GeoDataFrame(gender, geometry=geom)
gender['pop'] = gender['pop'].astype(int)
gender = gpd.sjoin(gender, study_area, "inner", predicate="within")
gender = gender[['peroid','id','areanm_left','gender','pop','geometry']]
gender.columns = ['분석기간','id','분석구간','성별','인구수','geometry']
#gender.to_file("D:/10_분석업무/구군청/26350_해운대구/2024년/20240704/26350_20240703.gpkg", layer="분석구간_성별_생활인구", driver="GPKG")
print()
ages = np.round(df.groupby(['peroid','id','areanm'])[['10less', '10ages', '20ages', '30ages', 
                                                      '40ages', '50ages', '60ages', '70more']].sum().reset_index().sort_values(by=['peroid','areanm']))
ages = pd.melt(ages, id_vars=['peroid', 'id', 'areanm'], value_vars=['10less', '10ages', '20ages', '30ages', 
                                                                     '40ages', '50ages', '60ages', '70more'], var_name='class', value_name='pop')
ages.loc[ages['class'].str.contains('10less|10ages'), 'age'] = "20대미만"
ages.loc[ages['class'].str.contains('20ages'), 'age'] = "20대"
ages.loc[ages['class'].str.contains('30ages'), 'age'] = "30대"
ages.loc[ages['class'].str.contains('40ages'), 'age'] = "40대"
ages.loc[ages['class'].str.contains('50ages'), 'age'] = "50대"
ages.loc[ages['class'].str.contains('60ages'), 'age'] = "60대"
ages.loc[ages['class'].str.contains('70more'), 'age'] = "70이상"
ages = ages[['peroid', 'id', 'areanm', 'age', 'pop']]
print("===== 연령 생활인구 밀도 시각화용 데이터 =====")
ages = pd.merge(ages, area_grid, "inner", "id")
ages = pd.DataFrame(ages.drop(columns=['areanm_y', 'geometry']))
ages.columns = ['peroid','id','areanm','age','pop','x','y']
geom = gpd.points_from_xy(ages.x, ages.y, crs="EPSG:5179")
ages = gpd.GeoDataFrame(ages, geometry=geom)
ages['pop'] = ages['pop'].astype(int)
ages = gpd.sjoin(ages, study_area, "inner", predicate="within")
ages = ages[['peroid','id','areanm_left','age','pop','geometry']]
ages.columns = ['분석기간','id','분석구간','연령','인구수','geometry']
ages.to_file("D:/10_분석업무/구군청/26350_해운대구/2024년/20240704/26350_20240703.gpkg", layer="분석구간_연령_생활인구", driver="GPKG")


===== 성별 생활인구 밀도 시각화용 데이터 =====

===== 연령 생활인구 밀도 시각화용 데이터 =====


#### 타지역, 외국인 방문인구 현황

In [7]:
print("====== 타지역 내국인 방문인구 현황 ======")
df = pd.read_csv("project/해운대축제_타지역_내국인 방문인구.csv", sep="|", parse_dates=['std_ymd'], dtype={'id': str, 'inflow_cd': str}, encoding="utf-8")
df['scode'] = df['inflow_cd'].str[:2]
df = pd.merge(df, area_grid, "inner", "id")
df = pd.DataFrame(df.drop(columns=['areanm_y','geometry']))
df = df[['std_ymd','id','areanm_x','scode','h_pop','w_pop','v_pop','x','y']]
df.columns = ['std_ymd','id','areanm','scode','h_pop','w_pop','v_pop','x','y']
df = pd.merge(df, hcode, "inner", "scode")
inner = df.groupby(['std_ymd','id','areanm','scode','sname','x','y'])[['h_pop','w_pop','v_pop']].sum().reset_index().sort_values(by=['areanm','scode'])
geom = gpd.points_from_xy(inner.x, inner.y, crs="EPSG:5179")
inner = gpd.GeoDataFrame(inner, geometry=geom)
inner = gpd.sjoin(inner, study_area, "inner", predicate="within")
inner.loc[inner['std_ymd'].astype(str).str.contains('2024-05-10|2024-05-11|2024-05-12|2024-05-13'), 'peroid'] = "축제2주전"
inner.loc[inner['std_ymd'].astype(str).str.contains('2024-05-24|2024-05-25|2024-05-26|2024-05-27'), 'peroid'] = "축제기간"
inner = np.round(inner.groupby(['peroid','areanm_left','sname'])[['h_pop','w_pop','v_pop']].sum()).reset_index().sort_values(['peroid','areanm_left'])
inner = pd.melt(inner, id_vars=['peroid','areanm_left','sname',], value_vars=['h_pop','w_pop','v_pop'], var_name='class', value_name='pop')
inner.loc[inner['class'].str.startswith('h'), 'regists'] = "주거"
inner.loc[inner['class'].str.startswith('w'), 'regists'] = "직장"
inner.loc[inner['class'].str.startswith('v'), 'regists'] = "방문"
inner = inner[['peroid','areanm_left','sname','regists','pop']]
inner.columns = ['분석기간','분석구간','유입지코드','생활목적','인구수']
inner.to_csv("project/분석구간_타지역_방문인구.csv", sep="|", encoding="utf-8", index=False)

====== 타지역 내국인 방문인구 현황 ======


In [13]:
print("========== 외국인 방문인구 현황 ==========")
df = pd.read_csv("project/해운대축제_전체_외국인 방문인구.csv", sep="|", parse_dates=['std_ymd'], dtype={'id': str}, encoding="utf-8")
df = pd.merge(df, area_grid, "inner", "id")
df = pd.DataFrame(df.drop(columns=['areanm_y','geometry']))
df = df[['std_ymd','id','areanm_x','mcc_nm','pop','x','y']]
df.columns = ['std_ymd','id','areanm','mcc_nm','pop','x','y']
outer = df.groupby(['std_ymd','areanm','mcc_nm','x','y'])[['pop']].sum().reset_index().sort_values(by=['std_ymd','areanm'])
geom = gpd.points_from_xy(outer.x, outer.y, crs="EPSG:5179")
outer = gpd.GeoDataFrame(outer, geometry=geom)
outer = gpd.sjoin(outer, study_area, "inner", predicate="within")
outer.loc[outer['std_ymd'].astype(str).str.contains('2024-05-10|2024-05-11|2024-05-12|2024-05-13'), 'peroid'] = "축제2주전"
outer.loc[outer['std_ymd'].astype(str).str.contains('2024-05-24|2024-05-25|2024-05-26|2024-05-27'), 'peroid'] = "축제기간"
outer = outer.groupby(['peroid','areanm_left','mcc_nm'])[['pop']].sum().reset_index().sort_values(by=['peroid','areanm_left'])
outer.columns = ['분석기간', '분석구간', '국적', '외국인수']
outer.to_csv("project/분석구간_국적별_외국인.csv", sep="|", encoding="utf-8", index=False)

========== 외국인 방문인구 현황 ==========
